In [31]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [32]:
# 1. GENERATE STORES (The Spoke)
retailers = ['Jewel-Osco', 'Mariano\'s', 'Target']
locations = ['Naperville', 'Lincoln Park', 'Evanston', 'Aurora', 'Oak Park', 'Schaumburg']

stores_data = []
num_stores = 50 #number of stores to consider in study
for i in range(num_stores):
    brand = np.random.choice(retailers)
    loc = np.random.choice(locations)
    # Estimate distance from Aurora, IL hub (usually 5 to 45 miles)
    dist = np.random.uniform(5, 45) 
    stores_data.append([f"STR_{i:04d}", f"{brand} - {loc} #{i}", dist])

df_stores = pd.DataFrame(stores_data, columns=['store_id', 'store_name', 'miles_from_hub'])
print (df_stores)

    store_id                     store_name  miles_from_hub
0   STR_0000        Mariano's - Oak Park #0       29.561427
1   STR_0001           Target - Oak Park #1       34.065077
2   STR_0002        Mariano's - Oak Park #2        9.788583
3   STR_0003       Target - Lincoln Park #3       27.797730
4   STR_0004    Mariano's - Lincoln Park #4       33.363486
5   STR_0005      Mariano's - Schaumburg #5       27.359968
6   STR_0006        Mariano's - Evanston #6       33.433731
7   STR_0007        Mariano's - Evanston #7       38.375975
8   STR_0008         Target - Schaumburg #8       18.283762
9   STR_0009   Jewel-Osco - Lincoln Park #9       25.468121
10  STR_0010      Jewel-Osco - Evanston #10       33.255463
11  STR_0011         Mariano's - Aurora #11       43.791773
12  STR_0012      Target - Lincoln Park #12       40.841029
13  STR_0013        Jewel-Osco - Aurora #13       21.234781
14  STR_0014      Jewel-Osco - Oak Park #14        8.077304
15  STR_0015            Target - Aurora 

In [33]:
# 2. GENERATE ASSETS (The Laundry Jugs)
# 2026 ESG Specs: Every jug has a 'birth weight' and material type
assets_data = []
num_of_jugs = 1000 #update to 5,000 and eventually 1,000,000 for pilot study
for i in range(num_of_jugs):
    assets_data.append([
        f"JUG_{i:05d}", 
        "HDPE_Plastic", 
        120, # grams of plastic
        "2025-11-01" # Launch date
    ])

df_assets = pd.DataFrame(assets_data, columns=['asset_id', 'material', 'weight_grams', 'initial_date'])

#print (df_assets) 


In [34]:
# 3. GENERATE LOGISTICS EVENTS (The Loop)
# This simulates the "Asset Management" problem: Where is the jug?
events = []
current_time = datetime(2025, 11, 1)

for jug_id in df_assets['asset_id']:
    # Simulate 3 to 10 rotations for each jug
    rotations = np.random.randint(3, 11)
    for r in range(rotations):
        store = df_stores.sample(1).iloc[0]
    
    # --- PHASE 1: AT RETAIL ---
    # Time spent sitting on the shelf at Target/Jewel
        shelf_life = np.random.randint(7, 31) 
        current_time += timedelta(days=shelf_life)
        events.append([jug_id, store['store_id'], 'purchased', current_time])
    
    # --- PHASE 2: AT HOME (Usage) ---
    # Based on your 73-day average calculation
        usage_life = np.random.randint(45, 91) 
        current_time += timedelta(days=usage_life)
    
    # --- PHASE 3: THE RETURN LAG ---
    # Time between 'Empty' and the customer actually going back to the store
        return_lag = np.random.randint(3, 15)
        current_time += timedelta(days=return_lag)
    
        if np.random.random() < 0.90: # 90% Return Rate
            events.append([jug_id, store['store_id'], 'returned_to_store', current_time])
        
        # Step C: Back to Aurora (Transit + Washing time)
            processing_time = np.random.randint(2, 6)
            current_time += timedelta(days=processing_time)
            events.append([jug_id, 'HUB_AURORA', 'washed', current_time])
        else:
            events.append([jug_id, 'UNKNOWN', 'lost_to_leakage', current_time])
            break   

df_events = pd.DataFrame(events, columns=['asset_id', 'location_id', 'event_type', 'timestamp'])

In [35]:
# SAVE DATA
df_stores.to_csv('stores.csv', index=False)
df_assets.to_csv('assets.csv', index=False)
df_events.to_csv('logistics_events.csv', index=False)

print("Project Data Generated: stores.csv, assets.csv, logistics_events.csv")

Project Data Generated: stores.csv, assets.csv, logistics_events.csv
